## config

In [7]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
import matplotlib.pyplot as plt
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA

import json
import glob

from googleapiclient.discovery import build

In [8]:
pd.set_option('display.float_format', '{:.4f}'.format)

In [13]:
with open ("../../performative.txt", "r") as f:
    lines = f.readlines()
    API_KEY = lines[2].strip()
    
yt = build("youtube", "v3", developerKey=API_KEY)

## data

In [9]:
df = pd.read_json("../../data/processed/ve_channels/ve.json")

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 252 entries, 0 to 251
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   channel_id    251 non-null    object
 1   channel_name  251 non-null    object
dtypes: object(2)
memory usage: 5.9+ KB


In [11]:
df.head()

,channel_id,channel_name
0,UCqgh7QNlVV8kv3LBTJZYWrg,The Film Essay
1,UCjFqcJQXGZ6T6sxyFB-5i6A,Every Frame a Painting
2,UCr8r7UVsDaRiIQSVUHVBd_A,Nikki Carreon
3,UCjKSoJlPgcK6BmoSqXpj5xQ,Action Button
4,UCUyvQV2JsICeLZP4c_h40kA,Thomas Flight


In [24]:
resp = yt.channels().list(
    part="brandingSettings,contentDetails,id,localizations,snippet,statistics,status,topicDetails",
    id="UCqgh7QNlVV8kv3LBTJZYWrg"
).execute()

print(json.dumps(resp["items"][0], indent=2))

{
  "kind": "youtube#channel",
  "etag": "N3Nhoav8KByMZ7nuavkOlXQgnE4",
  "id": "UCqgh7QNlVV8kv3LBTJZYWrg",
  "snippet": {
    "title": "The Film Essay",
    "description": "A video essay channel devoted to making videos exploring cinema as a medium, the techniques used and all that lies beyond.\nLike me on Facebook!    https://www.facebook.com/Thefilmessay/?ref=aymt_homepage_panel\n\nFor all business-related enquires: thefilmessay@gmail.com\n",
    "customUrl": "@thefilmessay9832",
    "publishedAt": "2016-08-16T14:09:38Z",
    "thumbnails": {
      "default": {
        "url": "https://yt3.ggpht.com/ytc/AIdro_lCt-zJAeXGbrNbcp9eMhre6sOT-_Omf7nvVuUdftpJBw=s88-c-k-c0x00ffffff-no-rj",
        "width": 88,
        "height": 88
      },
      "medium": {
        "url": "https://yt3.ggpht.com/ytc/AIdro_lCt-zJAeXGbrNbcp9eMhre6sOT-_Omf7nvVuUdftpJBw=s240-c-k-c0x00ffffff-no-rj",
        "width": 240,
        "height": 240
      },
      "high": {
        "url": "https://yt3.ggpht.com/ytc/AIdro_l

feature columns

In [ ]:
channel = resp["items"][0]
channel["snippet"]["title"]
channel["snippet"]["description"]
channel["snippet"]["country"]
channel["topicDetails"]["topicCategories"]
channel["brandingSettings"]["channel"]["keywords"]
videos = channel["contentDetails"]["relatedPlaylists"]["uploads"]

'UUqgh7QNlVV8kv3LBTJZYWrg'

In [ ]:
resp_vids = yt.playlistItems().list(
    part="contentDetails,id,snippet,status",
    playlistId=videos, 
).execute()

In [49]:
print(json.dumps(resp_vids["items"][0], indent=2))

{
  "kind": "youtube#playlistItem",
  "etag": "SDGLDt-RgAs0tvD4PvWKtlOpkjE",
  "id": "VVVxZ2g3UU5sVlY4a3YzTEJUSlpZV3JnLmlBUDZIbXA4N1hN",
  "snippet": {
    "publishedAt": "2026-02-06T17:13:58Z",
    "channelId": "UCqgh7QNlVV8kv3LBTJZYWrg",
    "title": "The Hidden Philosophy Lesson in The Before Trilogy (Richard Linklater)",
    "description": "Richard Linklater\u2019s Before trilogy and Alain de Botton\u2019s Essays in Love both peel back the fantasy of romance to reveal something deeper: love as a lifelong negotiation between two imperfect people. \n\n#beforesunrise #beforesunset #beforemidnight #richardlinklater #EssaysInLove #alaindebotton  #philosophy   #loveexplained  #movieanalysis #relationshipadvice #cinemalovers  #shorts",
    "thumbnails": {
      "default": {
        "url": "https://i.ytimg.com/vi/iAP6Hmp87XM/default.jpg",
        "width": 120,
        "height": 90
      },
      "medium": {
        "url": "https://i.ytimg.com/vi/iAP6Hmp87XM/mqdefault.jpg",
        "width":